<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px; border-radius: 16px; text-align: center; margin-bottom: 20px;">
  <h1 style="color: #e94560; font-size: 2.6em; margin-bottom: 8px; font-weight: 800;">LLM Engineering  &mdash; 01</h1>
  <h2 style="color: #a8dadc; font-size: 1.5em; font-weight: 400; margin-bottom: 20px;">From Zero to Agentic AI in 5 Stages</h2>
  <p style="color: #ccc; font-size: 1.05em; max-width: 650px; margin: 0 auto; line-height: 1.7;">
    A self-contained, hands-on tutorial. You will go from your very first API call to building a 
    multi-model, streaming, agentic AI pipeline &mdash; all in one notebook.
  </p>
</div>

---

## How This Notebook Is Structured

This notebook teaches through **5 progressive stages**. Each stage:
- Opens with a **"Why This Matters"** section connecting theory to real engineering
- Contains **deeply annotated code** you can read like a textbook
- Ends with a **Challenge** that pushes you to apply what you learned

| # | Stage | New Skills | Capstone You Will Build |
|---|-------|------------|------------------------|
| 1 | **The API Contract** | HTTP mechanics, tokens, cost math | Cricket Match Analyst |
| 2 | **Prompt Engineering as Code** | System/user/assistant roles, output formats, few-shot | Financial News Parser + Sentiment Engine |
| 3 | **The Multi-Provider World** | OpenAI, Gemini, Ollama, DeepSeek — one interface | Universal LLM Wrapper |
| 4 | **Memory, Context & Conversation** | Statelessness, history management, compression | Production Chatbot Class |
| 5 | **Agentic Pipelines** | Multi-step chains, LLM-as-router, streaming | Startup Intelligence Radar |

### Before You Start
- [ ] Virtual environment is active &mdash; you see `(.venv)` in your terminal
- [ ] `.env` file contains `OPENAI_API_KEY=sk-proj-...`
- [ ] Kernel is set to `.venv / Python 3.12.x` (top-right in VS Code or JupyterLab)
- [ ] Run cells **top to bottom** using `Shift + Enter`

> **Cost note:** This entire notebook costs under $0.10 to run end-to-end using `gpt-4.1-mini`. Free alternatives via Ollama are shown in Stage 3.

---
<div style="background: linear-gradient(90deg, #2d6a4f, #1b4332); padding: 18px 24px; border-radius: 10px;">
  <h1 style="color: #95d5b2; margin: 0;">Stage 1 &mdash; The API Contract</h1>
  <p style="color: #b7e4c7; margin: 8px 0 0 0; font-size: 1.05em;">Understanding exactly what happens when you call an LLM</p>
</div>

## Why This Matters

Every beginner treats an LLM like a magic box. Every professional treats it like a **web service**.

Understanding the exact mechanics &mdash; the HTTP verb, the headers, the JSON payload, the response schema &mdash; is what separates engineers who *use* AI from engineers who *build with* AI.

Before a single line of the `openai` library, we will understand what that library is actually doing.

---

## 1.1 What Is a Frontier Model?

A **frontier model** is an LLM at or near the cutting edge of capability. Examples today: GPT-4.1, Claude 3.5 Sonnet, Gemini 2.5 Pro.

Key architecture fact: **you don't run these models**. They live on massive GPU clusters owned by tech companies. You communicate with them over HTTPS, exactly as you would communicate with any REST API.

```
Your Python script  -->  HTTPS POST  -->  api.openai.com  -->  GPU cluster  -->  JSON response  -->  Your script
```

## 1.2 Tokens: The Atomic Unit of LLMs

LLMs don't process characters, words, or sentences. They process **tokens**.

A token is a sub-word chunk. In English, roughly 1 token = 4 characters = 0.75 words. But:
- `"hello"` = 1 token
- `"anthropomorphize"` = 4 tokens  
- `"मुंबई"` (Mumbai in Hindi) = 5+ tokens  

**Why tokens matter to you as an engineer:**
1. **Billing** &mdash; you pay per 1,000 or 1,000,000 tokens
2. **Context limits** &mdash; each model has a maximum context window in tokens
3. **Speed** &mdash; more tokens = slower response = higher latency

Quick pricing benchmark (mid-2025):
| Model | Input (per 1M tokens) | Output (per 1M tokens) |
|-------|----------------------|------------------------|
| gpt-4.1-nano | $0.10 | $0.40 |
| gpt-4.1-mini | $0.40 | $1.60 |
| gpt-4.1 | $2.00 | $8.00 |

A typical summarisation call uses ~500 tokens. At gpt-4.1-mini prices: **$0.0003 per call**.

**Note:** Check latest pricing for OpenAI models at: https://developers.openai.com/api/docs/pricing

### Why the following code cell is needed?
When the notebook runs from `notebooks/`, Python treats that folder as the current working directory.
That means it does not automatically find the sibling `utils/` package in the repo root.
By inserting the parent folder into `sys.path`, the notebook can import `utils.scrapper` normally.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd().parent
sys.path.insert(0, str(root))

In [ ]:
# ── Stage 1 ── Cell 1: Environment Setup ─────────────────────────────────────
# Always run this cell first. It loads your API key and imports all libraries.
print("Setting up environment and validating API key...\n")
import os, json, time, textwrap
import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

# Validation with helpful error messages
checks = {
    "Key exists":        bool(api_key),
    "Correct prefix":    (api_key or "").startswith("sk-proj-"),
    "No whitespace":     api_key == (api_key or "").strip(),
}
for check, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {check}")

if all(checks.values()):
    print("\nAll checks passed! Ready to build.")
else:
    print("\nFix the issues above before continuing.")

## 1.3 The Raw HTTP Contract

The `openai` Python package weighs about 1 MB. There is zero model code inside it. It is a **thin wrapper** that:
1. Takes your Python objects
2. Serialises them to JSON
3. Sends an HTTPS POST with an Authorization header
4. Deserialises the JSON response back into Python objects

Let's prove this by calling the endpoint directly with `requests`.

In [ ]:
# ── Stage 1 ── Cell 2: The Raw HTTP Call ────────────────────────────────────
# This is what the openai library does under the hood, spelled out explicitly.

ENDPOINT = "https://api.openai.com/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {api_key}",   # Bearer token auth - standard OAuth 2.0
    "Content-Type":  "application/json",    # we're sending JSON
}

payload = {
    "model": "gpt-4.1-nano",               # cheapest/fastest model for demos
    "messages": [
        {"role": "user", "content": "In one sentence: what is a large language model?"}
    ],
    "max_tokens": 100,                     # cap the response length
    "temperature": 0.3,                    # lower = more deterministic, higher = more creative
}

# Time the call to build intuition about latency
t0 = time.time()
resp = requests.post(ENDPOINT, headers=headers, json=payload)
latency_ms = (time.time() - t0) * 1000

print(f"HTTP Status  : {resp.status_code} ({'OK' if resp.status_code == 200 else 'ERROR'})")
print(f"Latency      : {latency_ms:.0f} ms")
print()

data = resp.json()

# The response schema — learn this cold, you'll see it everywhere
print("Response schema keys:", list(data.keys()))
print()
print("Usage (what you're billed for):")
usage = data.get("usage", {})
print(f"  prompt_tokens     : {usage.get('prompt_tokens')}")
print(f"  completion_tokens : {usage.get('completion_tokens')}")
print(f"  total_tokens      : {usage.get('total_tokens')}")
print()

# Cost calculation
cost = usage.get("total_tokens", 0) / 1_000_000 * 0.15
print(f"Estimated cost   : ${cost:.6f}")
print()
print("The answer:")
print(data["choices"][0]["message"]["content"])

In [ ]:
# ── Stage 1 ── Cell 3: The openai Library (Same Call, Nicer Syntax) ──────────
# Now use the library. Notice it's the same parameters, just Pythonic.

openai = OpenAI()   # reads OPENAI_API_KEY from environment automatically

response = openai.chat.completions.create(
    model       = "gpt-4.1-nano",
    messages    = [{"role": "user", "content": "In one sentence: what is a large language model?"}],
    max_tokens  = 100,
    temperature = 0.3,
)

# The response is a Pydantic model with attribute access instead of dict["key"]
print("Model used    :", response.model)
print("Finish reason :", response.choices[0].finish_reason)
print("  'stop'   = finished naturally")
print("  'length' = cut off by max_tokens")
print()
print("Prompt tokens     :", response.usage.prompt_tokens)
print("Completion tokens :", response.usage.completion_tokens)
print()
print("Answer:", response.choices[0].message.content)

# KEY INSIGHT: response.choices is a LIST — models can return multiple completions
# (controlled by n= parameter). You almost always just use choices[0].

## 1.4 Building the Cricket Match Analyst

Now we apply what we know to a real project. The pattern is:

```
Scorecard text  →  truncate to fit context  →  craft prompt  →  call API  →  render
```

This is the **foundational pipeline** that everything else in the course builds on.

Cricket is a data-rich sport — each match produces scores, partnerships, economy rates, and narrative arcs. Feeding raw scorecard data to an LLM and getting expert-level analysis back is a perfect first capstone: real data, real structure, genuinely useful output.

In [ ]:
# ── Stage 1 ── Cell 4: Preparing Cricket Scorecard Data ─────────────────────
# We use a realistic IPL match scorecard as our raw text input.
# In production you'd scrape this from Cricinfo/ESPNcricinfo; here we use a
# hard-coded example so the notebook runs without network dependency.

SAMPLE_SCORECARD = """
IPL 2024 Final — Kolkata Knight Riders vs Sunrisers Hyderabad
Venue: MA Chidambaram Stadium, Chennai | Date: 26 May 2024

KKR INNINGS: 240/6 (20 overs)
Phil Salt        c Abhishek b Cummins    54 (27)  6x4 4x6
Sunil Narine     b Shahbaz              52 (24)  4x4 5x6
Angkrish Raghuvanshi c Kamindu b Cummins 16 (9)
Shreyas Iyer     not out               24 (13)  2x4 2x6
Andre Russell    b Jaydev              19 (8)   1x4 2x6
Ramandeep Singh  not out               11 (5)

Extras: 12 (wd 9, nb 3)

BOWLING (SRH):
Pat Cummins      4-0-42-2  Economy: 10.5
Bhuvneshwar Kumar 4-0-51-0  Economy: 12.75
Shahbaz Ahmed    4-0-33-1  Economy: 8.25
Jaydev Unadkat   4-0-40-1  Economy: 10.0
T Natarajan      4-0-61-0  Economy: 15.25

SRH INNINGS: 113 all out (18.3 overs)
Abhishek Sharma  c Salt b Varun        4 (5)
Travis Head      b Narine             14 (9)   1x4 1x6
Rahul Tripathi   c Iyer b Chakravarthy 10 (11)
Aiden Markram    run out              20 (15)  2x4 1x6
Heinrich Klaasen lbw b Harshit        5 (6)
Pat Cummins      c Russell b Varun    24 (15)  1x4 2x6

BOWLING (KKR):
Varun Chakravarthy 4-0-28-3  Economy: 7.0
Sunil Narine       3.3-0-19-2  Economy: 5.43
Mitchell Starc     4-0-22-2  Economy: 5.5
Harshit Rana       4-0-27-2  Economy: 6.75

Result: KKR won by 127 runs
Player of the Match: Sunil Narine (52 off 24 + 2 wickets)
"""

print(f"Scorecard loaded: {len(SAMPLE_SCORECARD):,} characters")
print()
print("First 300 characters preview:")
print("-" * 50)
print(SAMPLE_SCORECARD[:300])

# WHY WE TRUNCATE: GPT-4.1-mini has a 1M token context window,
# but we don't need to send the entire scorecard verbatim.
# 3,000 chars ~ 750 tokens ~ $0.0003 at gpt-4.1-mini prices — more than enough for rich analysis.

In [ ]:
# ── Stage 1 ── Cell 5: Prompt Design for Cricket Analysis ────────────────────
# We'll explore prompts deeply in Stage 2. For now, observe the structure.

# The SYSTEM prompt sets the AI's "job description" and output requirements
CRICKET_ANALYST_SYSTEM = """You are a professional cricket analyst with deep expertise in T20 cricket.
You produce sharp, insightful match analysis for fans and coaches alike.

Your match reports must:
- Start with a 1-sentence "Match Verdict" capturing the story of the game
- Section 1 — Key Performances: top 3 individual contributions (bat + ball), each with a 1-line insight
- Section 2 — Turning Point: identify the exact over/moment that decided the match
- Section 3 — Numbers That Matter: 3 statistics that tell the real story (not the obvious ones)
- Section 4 — Verdict: one paragraph, written like a cricket journalist
- Be written in markdown
- Never state the obvious — every line must add insight beyond the raw numbers"""

# The USER prompt provides the actual content to process
def build_analyst_messages(scorecard: str, char_limit: int = 3000) -> list:
    truncated = scorecard[:char_limit]
    
    user_msg = f"""Please analyse this cricket match scorecard.

Character count sent: {len(truncated):,} of {len(scorecard):,} total

Scorecard:
{truncated}"""
    
    return [
        {"role": "system", "content": CRICKET_ANALYST_SYSTEM},
        {"role": "user",   "content": user_msg},
    ]

# Preview the messages we'll send
msgs = build_analyst_messages(SAMPLE_SCORECARD)
print(f"Messages built: {len(msgs)} messages")
print(f"Total chars in payload: {sum(len(m['content']) for m in msgs):,}")

In [ ]:
# ── Stage 1 ── Cell 6: The Complete Cricket Analyst ──────────────────────────

def analyse_match(scorecard: str, model: str = "gpt-4.1-mini") -> str:
    """Take a raw scorecard string and return an AI-generated match analysis."""
    messages = build_analyst_messages(scorecard)
    response = openai.chat.completions.create(
        model       = model,
        messages    = messages,
        temperature = 0.2,   # low temperature = consistent, factual output
    )
    return response.choices[0].message.content

def display_analysis(scorecard: str, model: str = "gpt-4.1-mini") -> None:
    """Analyse a scorecard and render the markdown output in the notebook."""
    print(f"Analysing match ({len(scorecard):,} chars of scorecard data)...")
    result = analyse_match(scorecard, model)
    display(Markdown(result))

# Try it!
display_analysis(SAMPLE_SCORECARD)

In [ ]:
# ── Stage 1 ── Cell 7: Multi-Match Comparison ────────────────────────────────
# A real use case: analyse multiple matches back-to-back for a series report.

MATCH_2_SCORECARD = """
IPL 2024 — Qualifier 1: Rajasthan Royals vs Kolkata Knight Riders
Venue: Narendra Modi Stadium, Ahmedabad | Date: 21 May 2024

RR INNINGS: 171/8 (20 overs)
Yashasvi Jaiswal  b Harshit          9  (11)
Jos Buttler       c Narine b Starc  39  (28)  4x4 2x6
Sanju Samson      run out           26  (23)  2x4 1x6
Riyan Parag       c Salt b Russell  32  (18)  2x4 2x6
Ravichandran Ashwin not out         18  (12)

BOWLING (KKR):
Mitchell Starc    4-0-35-2  Economy: 8.75
Sunil Narine      4-0-28-1  Economy: 7.0
Harshit Rana      4-0-40-2  Economy: 10.0

KKR INNINGS: 174/4 (19.4 overs)
Phil Salt         b Ashwin          38  (23)
Sunil Narine      b Chahal          23  (13)
Angkrish Raghuvanshi not out        31  (20)
Shreyas Iyer      not out           16  (9)

Result: KKR won by 6 wickets
Player of the Match: Mitchell Starc
"""

scorecards_to_compare = [
    ("IPL Final 2024", SAMPLE_SCORECARD),
    ("IPL Qualifier 1 2024", MATCH_2_SCORECARD),
]

for match_name, scorecard in scorecards_to_compare:
    print(f"\n{'='*60}")
    print(f"MATCH: {match_name}")
    display_analysis(scorecard)

## Stage 1 Challenge

**Build a Tournament Summary Generator**

You are a sports data analyst. A cricket board wants an automated end-of-round tournament digest.

Your task:
1. Create a `TOURNAMENT_SYSTEM` prompt that formats output as: `Match | Winner | Star Performer | One-liner`
2. Create a `tournament_digest(scorecards: list)` function that analyses each match and returns a combined digest
3. Include a cost estimate at the end (calculate tokens used across all calls)

**Bonus challenge:** Add a `leaderboard=True` parameter that adds a final LLM call to rank the top 5 players across all matches based on the analyses.

In [ ]:
# ── Stage 1 ── Challenge: Tournament Summary Generator ───────────────────────

TOURNAMENT_SYSTEM = """TODO: Write a system prompt that produces a structured match digest.
Format: | Match | Winner | Star Performer | One-liner insight |
Be concise — one row per match. Prioritise surprising or insightful observations."""

def tournament_digest(scorecards: list, model: str = "gpt-4.1-mini") -> None:
    """
    Generate and display a structured tournament digest for a list of scorecards.
    
    Parameters:
    - scorecards: list of (match_name, scorecard_text) tuples
    - model: the LLM to use
    """
    # TODO:
    # 1. For each scorecard, call the analyst with your TOURNAMENT_SYSTEM prompt
    # 2. Track total tokens used across all calls (hint: response.usage.total_tokens)
    # 3. Calculate and print total cost at the end
    # 4. BONUS: add a leaderboard synthesis call across all analyses
    
    total_tokens = 0
    results = []
    
    for match_name, scorecard in scorecards:
        # YOUR CODE HERE
        pass
    
    # Print cost summary
    cost = total_tokens / 1_000_000 * 0.40  # gpt-4.1-mini input rate
    print(f"\nTotal tokens used: {total_tokens:,}")
    print(f"Estimated cost: ${cost:.5f}")


# Test it (uncomment when ready)
# tournament_digest([
#     ("IPL Final 2024", SAMPLE_SCORECARD),
#     ("IPL Qualifier 1 2024", MATCH_2_SCORECARD),
# ])

---
<div style="background: linear-gradient(90deg, #7b2d8b, #4a0e8f); padding: 18px 24px; border-radius: 10px;">
  <h1 style="color: #e0aaff; margin: 0;">Stage 2 &mdash; Prompt Engineering as Code</h1>
  <p style="color: #c77dff; margin: 8px 0 0 0; font-size: 1.05em;">Prompts are not instructions. They are programs.</p>
</div>

## Why This Matters

Amateur AI users write prompts. Professional AI engineers **design prompt systems**.

The difference: an amateur asks "summarise this". A professional specifies:
- **Who** the model is (system role)
- **What format** it must return (output schema)  
- **What not to do** (negative constraints)
- **An example** of exactly right output (few-shot)

This stage teaches you to think about prompts the same way you think about function signatures.

---

## 2.1 The Three-Role Architecture

Every Chat Completions API call has up to three roles:

| Role | Who Writes It | Purpose | Analogy |
|------|--------------|---------|---------|
| `system` | You (developer) | Defines the AI's persona, task, output format, constraints | Job description |
| `user` | End user (or you simulating one) | The actual input to process | Customer request |
| `assistant` | AI (or you constructing history) | Previous AI responses — creates memory illusion | Prior work in a thread |

**Critical insight:** Models are trained to treat `system` with higher authority than `user`. When they conflict, `system` usually wins. This is how you enforce output format even when users try to override it.

## 2.2 Temperature and the Creativity Dial

`temperature` controls randomness in token selection:
- `0.0` — deterministic, always picks the most likely token. Best for: data extraction, structured output
- `0.7` — balanced. Best for: summarisation, Q&A
- `1.2+` — highly creative/random. Best for: brainstorming, creative writing

In [ ]:
# ── Stage 2 ── Cell 1: Temperature Experiment ────────────────────────────────
# See concretely how temperature changes output.

from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv(override=True)
openai = OpenAI()

prompt = "Complete this sentence with exactly 8 words: The future of AI will"

print("Same prompt, three temperatures:\n")
for temp in [0.0, 0.7, 1.4]:
    response = openai.chat.completions.create(
        model       = "gpt-4.1-nano",
        messages    = [{"role": "user", "content": prompt}],
        temperature = temp,
        max_tokens  = 20,
    )
    print(f"  temp={temp:.1f}: {response.choices[0].message.content.strip()}")

print()
print("Run this cell multiple times. temp=0.0 should be identical each run.")
print("temp=1.4 should vary more.")

## 2.3 Output Format Control

One of the most powerful prompt engineering techniques is **specifying output format precisely**. When you need machine-parseable output, use `response_format={"type": "json_object"}` alongside a schema in the system prompt.

This is essential for any production system where the LLM output feeds downstream code — such as a financial news parser that feeds a trading dashboard.

In [ ]:
# ── Stage 2 ── Cell 2: Structured JSON Extraction ───────────────────────────
# A real production use case: extract structured data from financial news headlines.
# This output would feed a trading dashboard, alert system, or risk model.

import json

FINANCIAL_NEWS_PARSER_SYSTEM = """You are a financial news parsing API used by a quantitative trading desk.
Extract structured data from raw financial news text.

Return ONLY valid JSON matching this exact schema — no explanation, no markdown fences:
{
  "headline": "string — cleaned, normalised headline",
  "companies_mentioned": ["list of ticker symbols or company names"],
  "sectors": ["list of affected sectors, e.g. Banking, IT, Energy"],
  "event_type": "one of: earnings | merger_acquisition | regulatory | macro | leadership | product | other",
  "sentiment": "one of: bullish | bearish | neutral",
  "sentiment_confidence": "number 0.0-1.0",
  "market_impact": "one of: high | medium | low",
  "key_figures": [{"entity": "string", "value": "string", "type": "revenue|profit|growth|price|other"}],
  "time_sensitivity": "one of: breaking | intraday | swing | long_term",
  "red_flags": ["any ambiguous language, potential misinformation, or data gaps — empty list if none"]
}"""

# Test news headlines
news_items = [
    """Reliance Industries Q4 net profit surges 12% to ₹21,243 crore, beats Street estimates.
    Jio platforms adds 8.1 million subscribers. Retail EBITDA up 18% YoY. 
    Board recommends dividend of ₹10 per share.""",
    
    """RBI keeps repo rate unchanged at 6.5% for 7th consecutive meeting.
    MPC vote was 4-2 in favour of status quo. Inflation outlook revised to 4.5%.
    Governor Das signals potential cut in Q3 if monsoon is normal.""",
]

for i, news in enumerate(news_items, 1):
    print(f"\n{'='*50}")
    print(f"News Item {i}:")
    print(news[:100] + "...")
    print()
    
    response = openai.chat.completions.create(
        model           = "gpt-4.1-mini",
        messages        = [
            {"role": "system", "content": FINANCIAL_NEWS_PARSER_SYSTEM},
            {"role": "user",   "content": f"Parse this financial news:\n{news}"},
        ],
        response_format = {"type": "json_object"},  # GUARANTEES valid JSON output
        temperature     = 0.1,  # low temp for consistent extraction
    )
    
    parsed = json.loads(response.choices[0].message.content)
    print(json.dumps(parsed, indent=2))

## 2.4 Few-Shot Prompting

**Zero-shot**: give no examples, just instructions  
**One-shot**: give one example of the desired input→output  
**Few-shot**: give 2-5 examples

Few-shot is one of the most reliable techniques for getting consistent format from an LLM. The model learns your pattern from examples rather than from abstract instructions.

When should you use it? When you have an unusual output format that's hard to describe in words, or when zero-shot keeps getting the format slightly wrong.

In [ ]:
# ── Stage 2 ── Cell 3: Few-Shot Prompting ────────────────────────────────────
# Teaching the model to classify support tickets via examples.

FEW_SHOT_SYSTEM = """Classify customer support tickets. 
Return exactly: CATEGORY | PRIORITY | ESTIMATED_RESOLUTION_TIME

Categories: billing | technical | account | feature_request | abuse
Priority: P0 (critical/data loss) | P1 (broken core feature) | P2 (degraded) | P3 (minor) | P4 (wish)

Examples:
---
Ticket: "My account was charged twice this month, $99 extra"
Output: billing | P1 | 4h
---
Ticket: "The entire platform is down, we can't access anything, our team is blocked"  
Output: technical | P0 | 1h
---
Ticket: "Would love a dark mode option"
Output: feature_request | P4 | backlog
---
Ticket: "I can't log in after resetting password, keeps saying invalid credentials"
Output: account | P1 | 2h
---"""

# Real tickets to classify
test_tickets = [
    "We accidentally deleted our production database. Need urgent help restoring.",
    "How do I export my data to CSV?",
    "The analytics dashboard is loading slowly, taking about 30 seconds",
    "I'd like the option to have weekly digest emails instead of daily",
    "Someone is using our brand name in their username, this seems like impersonation",
]

print("Classifying support tickets...\n")
print(f"{'Ticket':<60} Classification")
print("-" * 100)

for ticket in test_tickets:
    response = openai.chat.completions.create(
        model       = "gpt-4.1-nano",   # fast + cheap for classification
        messages    = [
            {"role": "system", "content": FEW_SHOT_SYSTEM},
            {"role": "user",   "content": ticket},
        ],
        temperature = 0.0,
        max_tokens  = 30,
    )
    classification = response.choices[0].message.content.strip()
    ticket_short = ticket[:57] + "..." if len(ticket) > 57 else ticket
    print(f"  {ticket_short:<60} {classification}")

## 2.5 The System Prompt as API Contract

Here is a powerful mental model: **treat your system prompt like a function signature**.

- The function name = the role you give the model
- Parameters = what you tell it about the input  
- Return type = the output format you specify
- Docstring = the constraints and edge cases

When you think this way, your prompts become more precise, more testable, and more maintainable.

In [ ]:
# ── Stage 2 ── Cell 4: Building a Sentiment Engine ──────────────────────────
# One input text, multiple output tones. Same user prompt, different system prompts.
# This demonstrates how system prompts function like configuration.

BASE_CONTENT = """
Q3 revenue declined 18% year-over-year to Rs 240 crore, primarily due to slower 
enterprise sales cycles and a one-time Rs 15 crore write-down on legacy inventory.
The company maintains Rs 80 crore in cash reserves and has secured a credit facility.
Management reaffirms full-year guidance contingent on Q4 recovery.
"""

TONE_CONFIGS = {
    "Executive Summary": {
        "system": (
            "You are a CFO writing for the board. Be direct, use bullet points, "
            "lead with impact. 3 bullets maximum. Use Indian business context (Rs, crore)."
        ),
        "model": "gpt-4.1-mini",
    },
    "Press Release": {
        "system": (
            "You are a PR professional. Frame challenges positively, emphasise resilience "
            "and future outlook. Use corporate language. 2 short paragraphs."
        ),
        "model": "gpt-4.1-mini",
    },
    "Slack Message to Team": {
        "system": (
            "You are a startup CEO writing to your team on Slack. Be honest, human, "
            "acknowledge the difficulty, focus on what we can control. Conversational tone. 3-4 sentences."
        ),
        "model": "gpt-4.1-mini",
    },
    "Reddit TL;DR": {
        "system": (
            "You write for a finance subreddit. Be blunt, slightly cynical, use "
            "casual internet language. One paragraph, max 60 words."
        ),
        "model": "gpt-4.1-mini",
    },
}

print("Same financial data, four different audiences:\n")
for tone_name, config in TONE_CONFIGS.items():
    response = openai.chat.completions.create(
        model       = config["model"],
        messages    = [
            {"role": "system", "content": config["system"]},
            {"role": "user",   "content": f"Rewrite this financial update:\n{BASE_CONTENT}"},
        ],
        temperature = 0.6,
    )
    print(f"{'='*60}")
    print(f"  [{tone_name}]")
    print(f"{'='*60}")
    print(response.choices[0].message.content.strip())
    print()

## Stage 2 Challenge

**Build a Prompt-Driven Data Pipeline**

You work at a fintech company. You receive raw transaction descriptions from bank statements and need to categorise and enrich them.

Given this raw transaction list:
```
"ZOMATO*ORDER 982736 MH"
"ATM WDL HDFC BANK BANDRA"  
"UPI/SWIGGY/INR/450.00"
"AMAZON PAY WALLET TOPUP"
"EMI HDFC CARLOAN 8899"
"NETFLIX.COM SUBSCRIPTION"
```

Build a `parse_transactions(transactions)` function that returns a JSON array where each item has:
- `description` (original text)
- `category` (food/transport/entertainment/finance/shopping/utilities)
- `merchant` (cleaned merchant name or null)
- `transaction_type` (debit/credit/transfer)
- `recurring` (true/false based on patterns like EMI, subscription)

Use few-shot prompting to achieve consistent output format.

In [ ]:
# ── Stage 2 ── Challenge: Transaction Parser ─────────────────────────────────

import json

TRANSACTION_SYSTEM = """TODO: Design a system prompt with few-shot examples that extracts:
- category, merchant, transaction_type, recurring
Return a JSON array. Ensure consistent format across all transactions."""

def parse_transactions(transactions: list) -> list:
    """
    Parse raw bank transaction descriptions into structured data.
    Returns a list of dicts with: description, category, merchant, 
    transaction_type, recurring
    """
    # TODO:
    # 1. Build the user prompt from the transactions list
    # 2. Call the API with response_format={"type": "json_object"}
    # 3. Parse and return the result
    pass

# Test data
raw_transactions = [
    "ZOMATO*ORDER 982736 MH",
    "ATM WDL HDFC BANK BANDRA",
    "UPI/SWIGGY/INR/450.00",
    "AMAZON PAY WALLET TOPUP",
    "EMI HDFC CARLOAN 8899",
    "NETFLIX.COM SUBSCRIPTION",
    "SALARY CREDIT TCS LTD",
    "PETROL PUMP HP ANDHERI",
]

# results = parse_transactions(raw_transactions)
# if results:
#     print(json.dumps(results, indent=2))

---
<div style="background: linear-gradient(90deg, #0077b6, #023e8a); padding: 18px 24px; border-radius: 10px;">
  <h1 style="color: #90e0ef; margin: 0;">Stage 3 &mdash; The Multi-Provider World</h1>
  <p style="color: #ade8f4; margin: 8px 0 0 0; font-size: 1.05em;">One interface. Multiple models. Intelligent routing.</p>
</div>

## Why This Matters

In production, you never rely on a single LLM provider. Reasons:

- **Cost optimisation**: route cheap tasks to cheap models, expensive tasks to powerful ones
- **Reliability**: if OpenAI goes down, failover to Gemini in under a second
- **Capability matching**: some models are better at code, others at reasoning, others at multilingual
- **Data sovereignty**: for sensitive data, use a local model that never leaves your infrastructure
- **Experimentation**: A/B test different models on the same prompts

## 3.1 The OpenAI-Compatible API Standard

OpenAI's Chat Completions API became the industry standard. Almost every major provider copied it:

```
POST {base_url}/chat/completions
Authorization: Bearer {api_key}
Body: {"model": "...", "messages": [...], ...}
```

The response schema is identical. This means you can switch providers by changing **two variables**: `base_url` and `api_key`.

| Provider | Base URL | Key Format |
|----------|----------|------------|
| OpenAI | `https://api.openai.com/v1` | `sk-proj-...` |
| Google Gemini | `https://generativelanguage.googleapis.com/v1beta/openai/` | `AIz...` |
| Ollama (local) | `http://localhost:11434/v1` | any string |
| Mistral | `https://api.mistral.ai/v1` | `...` |
| Together AI | `https://api.together.xyz/v1` | `...` |

In [ ]:
# ── Stage 3 ── Cell 1: Universal LLM Client Factory ──────────────────────────
# Build a clean abstraction that works with any provider.

import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv(override=True)

def make_client(provider: str) -> tuple:
    """
    Returns (client, default_model) for a given provider name.
    This is the factory pattern applied to LLM clients.
    """
    configs = {
        "openai": {
            "base_url": None,   # uses default OpenAI endpoint
            "api_key":  os.getenv("OPENAI_API_KEY"),
            "model":    "gpt-4.1-mini",
        },
        "gemini": {
            "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
            "api_key":  os.getenv("GOOGLE_API_KEY", "not-set"),
            "model":    "gemini-2.5-flash",
        },
        "ollama": {
            "base_url": "http://localhost:11434/v1",
            "api_key":  "ollama",   # Ollama ignores this, but the client requires a value
            "model":    "llama3.2",
        },
        "deepseek-local": {
            "base_url": "http://localhost:11434/v1",
            "api_key":  "ollama",
            "model":    "deepseek-r1:1.5b",
        },
    }
    
    cfg = configs.get(provider)
    if not cfg:
        raise ValueError(f"Unknown provider: {provider}. Choose from: {list(configs.keys())}")
    
    kwargs = {"api_key": cfg["api_key"]}
    if cfg["base_url"]:
        kwargs["base_url"] = cfg["base_url"]
    
    return OpenAI(**kwargs), cfg["model"]

# Create our primary client
openai_client, openai_model = make_client("openai")
print(f"Primary client ready: {openai_model}")
print()
print("Available providers: openai, gemini, ollama, deepseek-local")

In [ ]:
# ── Stage 3 ── Cell 2: Provider-Agnostic Ask Function ────────────────────────
# The heart of the multi-provider pattern: a single function that works with any client.

def ask(
    prompt:      str,
    system:      str   = "You are a helpful assistant.",
    provider:    str   = "openai",
    model:       str   = None,   # None = use provider default
    temperature: float = 0.7,
    max_tokens:  int   = 500,
) -> dict:
    """
    Universal LLM call. Works with any OpenAI-compatible provider.
    Returns a dict with: content, model, tokens, cost_estimate_usd
    """
    client, default_model = make_client(provider)
    model = model or default_model
    
    t0 = __import__("time").time()
    response = client.chat.completions.create(
        model       = model,
        messages    = [
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt},
        ],
        temperature = temperature,
        max_tokens  = max_tokens,
    )
    latency = __import__("time").time() - t0
    
    # Cost rates (per 1M tokens, approximate mid-2025)
    cost_rates = {
        "gpt-4.1-nano": 0.10, "gpt-4.1-mini": 0.40, "gpt-4.1": 2.00,
        "gemini-2.5-flash": 0.075, "llama3.2": 0.0, "deepseek-r1:1.5b": 0.0,
    }
    rate = cost_rates.get(model, 0.40)
    tokens = response.usage.total_tokens if response.usage else 0
    cost   = (tokens / 1_000_000) * rate
    
    return {
        "content":           response.choices[0].message.content,
        "model":             response.model,
        "tokens":            tokens,
        "latency_ms":        round(latency * 1000),
        "cost_estimate_usd": round(cost, 6),
    }

# Test it
result = ask("What is the most important skill for an AI engineer?", provider="openai")
print(f"Model   : {result['model']}")
print(f"Tokens  : {result['tokens']}")
print(f"Latency : {result['latency_ms']} ms")
print(f"Cost    : ${result['cost_estimate_usd']}")
print()
print(result["content"])

In [ ]:
# ── Stage 3 ── Cell 3: Google Gemini (Optional) ───────────────────────────────
# Get a free API key at: https://aistudio.google.com/api-keys
# Add to .env: GOOGLE_API_KEY=AIz...

import os
google_key = os.getenv("GOOGLE_API_KEY")

if not google_key or google_key == "not-set":
    print("No GOOGLE_API_KEY found. Skipping Gemini.")
    print("To try Gemini: get a free key at https://aistudio.google.com/api-keys")
    print("Then add to .env: GOOGLE_API_KEY=AIz...")
else:
    print("Google API key found! Testing Gemini...")
    result = ask(
        "In exactly 2 sentences: how does Gemini differ from GPT architecturally?",
        provider = "gemini"
    )
    print(f"Model  : {result['model']}")
    print(f"Tokens : {result['tokens']}")
    print()
    print(result["content"])
    print()
    print("Notice: same ask() function, completely different model family.")

In [ ]:
# ── Stage 3 ── Cell 4: Ollama Local Models ────────────────────────────────────
# Ollama lets you run open-source LLMs locally. 
# Install: https://ollama.com
# Then: ollama pull llama3.2

import requests as req

try:
    r = req.get("http://localhost:11434", timeout=2)
    ollama_running = "running" in r.text.lower()
    print(f"Ollama status: {r.text.strip()}")
except Exception:
    ollama_running = False
    print("Ollama is not running.")
    print("To start: open a terminal and run 'ollama serve'")
    print("To install: https://ollama.com")

if ollama_running:
    # Pull the model (downloads ~2GB once, cached after)
    import subprocess
    print("\nPulling llama3.2 (skip if already downloaded)...")
    # subprocess.run(["ollama", "pull", "llama3.2"], check=True)
    
    result = ask(
        "Explain in 2 sentences why running LLMs locally matters for privacy.",
        provider = "ollama",
        model    = "llama3.2",
    )
    print(f"\nModel  : {result['model']}")
    print(f"Cost   : ${result['cost_estimate_usd']} (it is free!)")
    print()
    print(result["content"])

In [ ]:
# ── Stage 3 ── Cell 5: DeepSeek-R1 (Reasoning Model) ─────────────────────────
# DeepSeek R1 is a reasoning model that "thinks" before answering.
# It shows its reasoning chain inside <think>...</think> tags.
# The 1.5B distilled version runs locally via Ollama.

if ollama_running:
    import subprocess
    print("Pulling deepseek-r1:1.5b...")
    # subprocess.run(["ollama", "pull", "deepseek-r1:1.5b"], check=True)
    
    reasoning_problem = """A hotel has 10 floors. There are 3 elevators.
    Each elevator can hold 8 people. The hotel has 200 guests.
    If all guests want to go to their rooms at the same time, 
    and each trip takes 2 minutes, how long until everyone is in their room?"""
    
    result = ask(
        reasoning_problem,
        provider    = "deepseek-local",
        max_tokens  = 800,
        temperature = 0.6,
    )
    print(f"Model: {result['model']}")
    print()
    print(result["content"])
    print()
    print("Notice the <think>...</think> block — visible chain-of-thought reasoning.")
else:
    print("Start Ollama to run DeepSeek locally.")

## 3.2 The Intelligent Router Pattern

In production systems, you don't just have one model &mdash; you **route** each request to the most appropriate model based on task characteristics. This is a real pattern used by companies like Notion, Cursor, and Perplexity.

In [ ]:
# ── Stage 3 ── Cell 6: Task-Aware Model Router ───────────────────────────────
# Route different task types to optimal models automatically.

TASK_ROUTING = {
    # task_type -> (provider, model, temperature, rationale)
    "classification":   ("openai", "gpt-4.1-nano",  0.0, "Fast, cheap, deterministic"),
    "extraction":       ("openai", "gpt-4.1-mini",  0.1, "Reliable JSON extraction"),
    "summarisation":    ("openai", "gpt-4.1-mini",  0.3, "Good quality, moderate cost"),
    "creative":         ("openai", "gpt-4.1",       0.9, "Best quality, higher cost"),
    "reasoning":        ("openai", "gpt-4.1-mini",  0.5, "Strong reasoning needed"),
    "local_private":    ("ollama", "llama3.2",       0.7, "Data never leaves machine"),
}

def smart_ask(prompt: str, task_type: str, system: str = "You are a helpful assistant.") -> dict:
    """
    Route a prompt to the optimal model for the task type.
    Prints routing decision for transparency.
    """
    if task_type not in TASK_ROUTING:
        task_type = "summarisation"   # safe default
    
    provider, model, temperature, rationale = TASK_ROUTING[task_type]
    
    # Fall back to openai if local not running
    if provider == "ollama" and not ollama_running:
        provider, model, temperature = "openai", "gpt-4.1-mini", 0.7
        rationale = "FALLBACK: Ollama unavailable, using OpenAI"
    
    print(f"  Routing to: {provider}/{model} ({rationale})")
    return ask(prompt, system=system, provider=provider, model=model, temperature=temperature)


# Demonstrate routing
tasks = [
    ("Is this text positive, negative, or neutral? Text: 'The product is fine.'", "classification"),
    ("What are 5 creative startup ideas combining AR and agriculture?", "creative"),
    ("Summarise the key points from this text in 3 bullets: LLMs are trained...", "summarisation"),
]

print("Smart routing demonstration:\n")
for prompt, task in tasks:
    print(f"Task type: {task}")
    result = smart_ask(prompt[:80] + "...", task)
    print(f"  Answer: {result['content'][:100]}...")
    print()

## Stage 3 Challenge

**Build a Model Benchmarking Tool**

Engineers often need to empirically compare models on their specific task, not rely on benchmarks designed by others.

Build a `benchmark(prompt, task_type, providers)` function that:
1. Sends the same prompt to each provider
2. Records: response content, latency, tokens, estimated cost
3. Uses an LLM judge (a separate GPT call) to score each response 1-10 on accuracy and clarity  
4. Prints a comparison table

This is a real pattern called "LLM-as-judge" used in production evaluation systems.

In [ ]:
# ── Stage 3 ── Challenge: Model Benchmarking Tool ────────────────────────────

JUDGE_SYSTEM = """You are an impartial AI response evaluator.
Given a prompt and a response, score the response from 1-10 on:
- Accuracy (is it correct and complete?)
- Clarity (is it well-written and easy to understand?)

Return ONLY JSON: {"accuracy": number, "clarity": number, "justification": "one sentence"}"""

def judge_response(prompt: str, response: str) -> dict:
    """Use an LLM to score a response. Returns accuracy/clarity scores."""
    import json
    result = openai_client.chat.completions.create(
        model           = "gpt-4.1-nano",
        messages        = [
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user", "content": f"Prompt: {prompt}\n\nResponse: {response}"},
        ],
        response_format = {"type": "json_object"},
        temperature     = 0.1,
    )
    return json.loads(result.choices[0].message.content)

def benchmark(prompt: str, providers_to_test: list = None) -> None:
    """
    Benchmark multiple providers on the same prompt.
    Displays a comparison table with scores.
    """
    if providers_to_test is None:
        providers_to_test = ["openai"]   # add "gemini", "ollama" if available
    
    print(f"Benchmarking: {prompt[:60]}...\n")
    results = []
    
    for provider in providers_to_test:
        print(f"  Testing {provider}...", end=" ", flush=True)
        try:
            r = ask(prompt, provider=provider, max_tokens=200)
            scores = judge_response(prompt, r["content"])
            results.append({
                "provider":   provider,
                "model":      r["model"],
                "latency_ms": r["latency_ms"],
                "tokens":     r["tokens"],
                "cost":       r["cost_estimate_usd"],
                "accuracy":   scores.get("accuracy", "?"),
                "clarity":    scores.get("clarity", "?"),
                "content":    r["content"][:80] + "...",
            })
            print(f"done ({r['latency_ms']}ms)")
        except Exception as e:
            print(f"ERROR: {e}")
    
    # Display results table
    print()
    print(f"{'Provider':<20} {'Model':<20} {'Latency':>8} {'Tokens':>7} {'Cost':>10} {'Accuracy':>9} {'Clarity':>8}")
    print("-" * 95)
    for r in results:
        print(f"{r['provider']:<20} {r['model']:<20} {r['latency_ms']:>7}ms {r['tokens']:>7} ${r['cost']:>9.5f} {r['accuracy']:>9} {r['clarity']:>8}")

# Test
# benchmark(
#     "Explain the difference between supervised and unsupervised learning in 3 sentences.",
#     providers_to_test=["openai"]  # add "gemini", "ollama" when available
# )

---
<div style="background: linear-gradient(90deg, #b5451b, #7d2210); padding: 18px 24px; border-radius: 10px;">
  <h1 style="color: #ffd6a5; margin: 0;">Stage 4 &mdash; Memory, Context &amp; Conversation</h1>
  <p style="color: #ffb347; margin: 8px 0 0 0; font-size: 1.05em;">The illusion of memory, and how to make it real.</p>
</div>

## Why This Matters

This is the stage where most AI beginners have their biggest misconception corrected.

**The misconception:** "The AI remembers our conversation."  
**The truth:** Every single API call is completely stateless. The model forgets everything the instant it responds.

**The engineering reality:** What looks like "memory" in ChatGPT is a product-level trick &mdash; the entire conversation history is stuffed into the `messages` array and re-sent with every single call. The model "remembers" because we show it the history, not because it stored anything.

As AI engineers, **implementing memory is entirely our responsibility**. This stage teaches you how.

---

## 4.1 Tokenisation Deep Dive

Before we can manage context, we need to understand what "context" actually costs.

The key library is **tiktoken** &mdash; OpenAI's tokeniser. It lets you count tokens before making API calls, enabling cost estimation, context budgeting, and intelligent truncation.

In [ ]:
# ── Stage 4 ── Cell 1: Tokenisation Mechanics ────────────────────────────────
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4.1-mini")

# Vocabulary size
print(f"Vocabulary size: {enc.n_vocab:,} tokens")
print()

# Tokenise different types of text
examples = [
    ("Hello world", "Simple English"),
    ("machine learning", "Common compound term"),
    ("supercalifragilisticexpialidocious", "Rare long word"),
    ("def fibonacci(n): return n if n<=1 else fibonacci(n-1)+fibonacci(n-2)", "Python code"),
    ("9.11 > 9.9", "Numeric comparison LLMs famously fail"),
    ("नमस्ते दुनिया", "Hindi — non-Latin scripts use more tokens per character"),
    ("SELECT * FROM users WHERE id = 1", "SQL"),
]

print(f"{'Text':<55} {'Tokens':>6}  Breakdown")
print("-" * 100)
for text, label in examples:
    tids  = enc.encode(text)
    parts = " | ".join(f"'{enc.decode([t])}'" for t in tids)
    print(f"  {text:<53} {len(tids):>6}  [{parts}]  ({label})")

print()
print("Key insight: '9.11' might tokenise as ['9', '.', '11'] or similar —")
print("the model sees numbers as token sequences, not as numeric values.")
print("This is why LLMs are unreliable at numeric comparison without reasoning steps.")

In [ ]:
# ── Stage 4 ── Cell 2: Context Window Calculator ─────────────────────────────
# Build a practical tool for context budget management.

import tiktoken

class ContextBudget:
    """
    Manages token budget for LLM API calls.
    Essential for production systems to avoid context overflow errors.
    """
    
    MODEL_LIMITS = {
        "gpt-4.1-nano":  1_000_000,
        "gpt-4.1-mini":  1_000_000,
        "gpt-4.1":       1_000_000,
        "llama3.2":        128_000,
        "deepseek-r1:1.5b": 64_000,
    }
    
    COST_PER_1M_TOKENS = {
        "gpt-4.1-nano": 0.10, "gpt-4.1-mini": 0.40, "gpt-4.1": 2.00,
        "llama3.2": 0.0, "deepseek-r1:1.5b": 0.0,
    }
    
    def __init__(self, model: str = "gpt-4.1-mini"):
        self.model = model
        self.enc   = tiktoken.encoding_for_model("gpt-4.1-mini")  # close enough for all
        self.max_tokens = self.MODEL_LIMITS.get(model, 128_000)
    
    def count(self, text: str) -> int:
        return len(self.enc.encode(text))
    
    def count_messages(self, messages: list) -> int:
        total = 0
        for msg in messages:
            total += self.count(msg.get("content", ""))
            total += 4  # overhead per message (role, formatting)
        return total + 2  # response prime
    
    def cost_estimate(self, messages: list, expected_output_tokens: int = 500) -> dict:
        input_tokens  = self.count_messages(messages)
        output_tokens = expected_output_tokens
        total_tokens  = input_tokens + output_tokens
        rate          = self.COST_PER_1M_TOKENS.get(self.model, 0.40)
        
        return {
            "model":            self.model,
            "context_limit":    f"{self.max_tokens:,}",
            "input_tokens":     input_tokens,
            "output_tokens":    f"~{output_tokens}",
            "total_tokens":     total_tokens,
            "pct_of_context":   f"{(input_tokens / self.max_tokens * 100):.2f}%",
            "cost_usd":         f"${(total_tokens / 1_000_000 * rate):.5f}",
            "fits_in_context":  input_tokens < self.max_tokens - 1000,
        }
    
    def truncate_to_fit(self, text: str, max_tokens: int) -> str:
        tokens    = self.enc.encode(text)
        if len(tokens) <= max_tokens:
            return text
        truncated = self.enc.decode(tokens[:max_tokens])
        return truncated + "\n\n[...content truncated to fit context window...]"


# Demo
budget = ContextBudget("gpt-4.1-mini")

long_document = "This is a very important document. " * 500
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user",   "content": long_document},
]

estimate = budget.cost_estimate(messages)
print("Context Budget Analysis:")
print("-" * 40)
for key, val in estimate.items():
    print(f"  {key:<22}: {val}")

print()
print(f"Original text: {budget.count(long_document):,} tokens")
truncated = budget.truncate_to_fit(long_document, 500)
print(f"After truncation: {budget.count(truncated):,} tokens")

## 4.2 Proving Statelessness

Let's demonstrate concretely that the model has absolutely no memory between calls.

In [ ]:
# ── Stage 4 ── Cell 3: The Statelessness Proof ───────────────────────────────
from openai import OpenAI
import os
load_dotenv = __import__("dotenv").load_dotenv
load_dotenv(override=True)
openai = OpenAI()

print("EXPERIMENT: Proving LLM statelessness\n")

# Call 1: Tell the model your name
r1 = openai.chat.completions.create(
    model    = "gpt-4.1-nano",
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": "Hi! My name is Priya and I'm a data engineer from Bangalore."},
    ],
)
print("CALL 1 (introducing ourselves):")
print(f"  User: Hi! My name is Priya and I'm a data engineer from Bangalore.")
print(f"  AI  : {r1.choices[0].message.content.strip()[:100]}...")
print()

# Call 2: FRESH call — no history — ask what it knows
r2 = openai.chat.completions.create(
    model    = "gpt-4.1-nano",
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": "What's my name and where am I from?"},  # new call, no history
    ],
)
print("CALL 2 (fresh call — no history passed):")
print(f"  User: What's my name and where am I from?")
print(f"  AI  : {r2.choices[0].message.content.strip()}")
print()
print("The AI has NO IDEA. Every call is fresh. Statelessness proven.")
print()

# Call 3: Include history — NOW it "remembers"
r3 = openai.chat.completions.create(
    model    = "gpt-4.1-nano",
    messages = [
        {"role": "system",    "content": "You are a helpful assistant."},
        {"role": "user",      "content": "Hi! My name is Priya and I'm a data engineer from Bangalore."},
        {"role": "assistant", "content": r1.choices[0].message.content},   # the AI's prior response
        {"role": "user",      "content": "What's my name and where am I from?"},
    ],
)
print("CALL 3 (history included — the trick revealed):")
print(f"  AI  : {r3.choices[0].message.content.strip()}")
print()
print("'Memory' = the entire prior conversation in the messages list. That's it.")

## 4.3 Building a Production Chatbot Class

Now we build the correct architecture: a class that manages conversation state, enforces context limits, and handles the history management that all chat applications need.

In [ ]:
# ── Stage 4 ── Cell 4: Production Chatbot Class ──────────────────────────────

import tiktoken
from openai import OpenAI
from datetime import datetime

class ProductionChatbot:
    """
    A production-ready chatbot that correctly implements conversation memory.
    
    Key features:
    - Maintains full conversation history
    - Tracks token usage across turns
    - Supports context compression when approaching limits
    - Logs all interactions with timestamps
    - Provides conversation export
    """
    
    def __init__(
        self,
        system_prompt:   str,
        model:           str   = "gpt-4.1-mini",
        max_history_tokens: int = 4000,  # compress history when it exceeds this
        temperature:     float = 0.7,
    ):
        self.model             = model
        self.temperature       = temperature
        self.max_history_tokens = max_history_tokens
        self.enc               = tiktoken.encoding_for_model("gpt-4.1-mini")
        self.client            = OpenAI()
        
        # Conversation state
        self.system_prompt = system_prompt
        self.history: list = []         # user/assistant turns only
        self.total_tokens_used: int = 0
        self.turn_count: int = 0
        self.session_start = datetime.now()
    
    def _count_tokens(self, text: str) -> int:
        return len(self.enc.encode(text))
    
    def _history_tokens(self) -> int:
        return sum(self._count_tokens(m["content"]) for m in self.history)
    
    def _build_messages(self) -> list:
        """Build the full messages list: system + history."""
        return [{"role": "system", "content": self.system_prompt}] + self.history
    
    def _maybe_compress_history(self):
        """
        If history is too large, summarise older turns to save tokens.
        This is the 'sliding window with summarisation' pattern.
        """
        if self._history_tokens() < self.max_history_tokens:
            return   # nothing to do
        
        # Keep the most recent 4 turns (2 user+assistant pairs)
        recent_turns = 8   
        if len(self.history) <= recent_turns:
            return
        
        # Summarise older turns
        old_turns    = self.history[:-recent_turns]
        recent_turns = self.history[-recent_turns:]
        
        old_text = "\n".join(f"{m['role'].upper()}: {m['content']}" for m in old_turns)
        
        summary_response = self.client.chat.completions.create(
            model    = "gpt-4.1-nano",   # use cheapest model for compression
            messages = [
                {"role": "system",  "content": "Summarise this conversation history concisely in 3-5 bullet points. Preserve all important facts, names, and decisions."},
                {"role": "user",    "content": old_text},
            ],
            temperature = 0.1,
            max_tokens  = 200,
        )
        summary = summary_response.choices[0].message.content
        
        # Replace old turns with a summary message
        compressed = {
            "role":    "system",
            "content": f"[CONVERSATION SUMMARY — {len(old_turns)//2} earlier turns]:\n{summary}"
        }
        self.history = [compressed] + recent_turns
        print(f"  [System: History compressed. {len(old_turns)} messages -> 1 summary]")
    
    def chat(self, user_message: str, verbose: bool = False) -> str:
        """Send a message and get a response. Manages all history automatically."""
        self.turn_count += 1
        
        # Add user message
        self.history.append({"role": "user", "content": user_message})
        
        # Compress history if needed
        self._maybe_compress_history()
        
        # Make the API call with full history
        messages = self._build_messages()
        
        if verbose:
            print(f"  Sending {len(messages)} messages ({self._history_tokens()} history tokens)")
        
        response = self.client.chat.completions.create(
            model       = self.model,
            messages    = messages,
            temperature = self.temperature,
        )
        
        reply = response.choices[0].message.content
        tokens_this_turn = response.usage.total_tokens
        self.total_tokens_used += tokens_this_turn
        
        # Add assistant response to history
        self.history.append({"role": "assistant", "content": reply})
        
        if verbose:
            cost = tokens_this_turn / 1_000_000 * 0.40
            print(f"  Response: {tokens_this_turn} tokens (${cost:.5f})")
        
        return reply
    
    def stats(self) -> dict:
        """Return session statistics."""
        session_mins = (datetime.now() - self.session_start).seconds / 60
        return {
            "turns":          self.turn_count,
            "history_msgs":   len(self.history),
            "history_tokens": self._history_tokens(),
            "total_tokens":   self.total_tokens_used,
            "session_mins":   round(session_mins, 1),
            "est_cost_usd":   f"${self.total_tokens_used / 1_000_000 * 0.40:.4f}",
        }
    
    def export(self) -> str:
        """Export conversation as markdown."""
        lines = [f"# Conversation Export\n", f"Model: {self.model}\n", "---\n"]
        for msg in self.history:
            role = "**You**" if msg["role"] == "user" else "**Assistant**"
            if msg["role"] == "system":
                role = "_[System]_"
            lines.append(f"{role}: {msg['content']}\n\n")
        return "".join(lines)


# Demo: 3-turn conversation
bot = ProductionChatbot(
    system_prompt = "You are a concise Python tutor. Give practical, brief answers with code examples when helpful.",
    model         = "gpt-4.1-mini",
    temperature   = 0.4,
)

questions = [
    "What's the difference between a list and a tuple in Python?",
    "When should I prefer one over the other?",
    "Can you give me a concrete real-world example where a tuple is the right choice?",
]

print("Multi-turn conversation with memory:\n")
for q in questions:
    print(f"You: {q}")
    reply = bot.chat(q, verbose=True)
    print(f"Bot: {reply[:200]}{'...' if len(reply)>200 else ''}\n")

print("\nSession stats:")
for k, v in bot.stats().items():
    print(f"  {k:<18}: {v}")

## Stage 4 Challenge

**Build an Interview Prep Bot**

Create a specialised chatbot for technical interview preparation.

Requirements:
1. System prompt: it's an interviewer for a specific role (e.g., "Senior Data Engineer at a fintech startup")
2. The bot asks questions one at a time, waits for the answer, then gives structured feedback
3. After 5 questions, it generates a final assessment report in JSON with: `{overall_score, strengths, gaps, recommendation}`
4. Track and display running token cost after each turn

**Advanced:** Add a "hint" feature where typing "hint" instead of an answer gives a nudge without revealing the full answer.

In [ ]:
# ── Stage 4 ── Challenge: Technical Interview Bot ────────────────────────────

INTERVIEWER_SYSTEM = """You are a senior technical interviewer at a fintech startup in Mumbai.
You are interviewing candidates for a Senior Data Engineer role.

Interview protocol:
1. Ask exactly ONE technical question at a time
2. After the candidate answers, give structured feedback: [SCORE: X/10] then 2-3 sentences
3. Then ask the next question
4. After 5 questions total, say INTERVIEW_COMPLETE and provide a final JSON assessment

Topics to cover: SQL/data modeling, Python for data, system design, cloud (AWS/GCP), 
data quality/testing

If the user types 'hint', give a small nudge without revealing the full answer.

Final assessment JSON format:
{
  "overall_score": number/10,
  "strengths": ["list"],
  "gaps": ["list"], 
  "recommendation": "strong_hire | hire | borderline | no_hire",
  "summary": "2-3 sentence summary"
}"""

class InterviewBot(ProductionChatbot):
    """Extends ProductionChatbot for structured interview sessions."""
    
    def __init__(self):
        super().__init__(
            system_prompt = INTERVIEWER_SYSTEM,
            model         = "gpt-4.1-mini",
            temperature   = 0.5,
        )
        self.question_count  = 0
        self.interview_done  = False
    
    def answer(self, response: str) -> str:
        """Submit an interview answer. Tracks question count."""
        reply = self.chat(response)
        
        if "INTERVIEW_COMPLETE" in reply:
            self.interview_done = True
            print("\nInterview complete! See assessment above.")
        else:
            self.question_count += 1
            cost = self.total_tokens_used / 1_000_000 * 0.40
            print(f"  [Question {self.question_count}/5 | Tokens: {self.total_tokens_used:,} | Cost: ${cost:.4f}]")
        
        return reply
    
    def start(self) -> str:
        """Begin the interview."""
        return self.chat("I'm ready for the interview. Please begin.")


# Interactive demo
# bot = InterviewBot()
# print("=== INTERVIEW BOT ===")
# print(bot.start())
# print()
# print("Type your answers. Type 'hint' for a hint. Run the cell repeatedly.")
# 
# # Example interaction:
# # print(bot.answer("An index is a data structure that speeds up queries by..."))
# # print(bot.answer("hint"))   # get a hint for the next question
print("InterviewBot class defined. Uncomment the code above to start an interview session.")

---
<div style="background: linear-gradient(90deg, #c1121f, #780000); padding: 18px 24px; border-radius: 10px;">
  <h1 style="color: #ffd6a5; margin: 0;">Stage 5 &mdash; Agentic Pipelines</h1>
  <p style="color: #ffb347; margin: 8px 0 0 0; font-size: 1.05em;">Multiple LLM calls. Autonomous decisions. Real business output.</p>
</div>

## Why This Matters

A single LLM call is a **function**. An **agentic pipeline** is a **program** where:

1. **Multiple LLM calls are chained** — the output of one is the input of the next
2. **LLMs make routing decisions** — not just generate text, but choose what to do next
3. **External tools are used** — web scraping, APIs, databases
4. **The system works autonomously** — no human intervention per step

This is the architectural pattern underlying: Cursor (coding AI), Perplexity (search AI), Notion AI (document AI), GitHub Copilot (code AI).

## 5.1 The Agentic Design Pattern

```
INPUT
  |
  v
[LLM: PLAN]    ← decides what to do
  |
  v
[TOOL: EXECUTE] ← does it (scrape, search, calculate, etc.)
  |
  v
[LLM: EVALUATE] ← judges if the result is good
  |
  v
[LLM: GENERATE] ← creates final output
  |
  v
OUTPUT (streamed)
```

Our Stage 5 project implements this pattern to build a **Startup Intelligence Radar** — a tool that researches any startup's website, classifies it, and produces a VC-style diligence brief. Investors, founders, and product teams all need this kind of signal extraction at scale.

In [ ]:
# ── Stage 5 ── Cell 1: Setup ──────────────────────────────────────────────────
import os, json, time
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from scraper import fetch_website_links, fetch_website_contents
import tiktoken

load_dotenv(override=True)
openai = OpenAI()

MODEL_FAST    = "gpt-4.1-nano"   # for routing decisions (cheap, fast)
MODEL_MAIN    = "gpt-4.1-mini"   # for main generation (quality/cost balance)

enc = tiktoken.encoding_for_model("gpt-4.1-mini")

api_key = os.getenv("OPENAI_API_KEY")
if api_key and api_key.startswith("sk-proj-"):
    print(f"Ready. Fast model: {MODEL_FAST} | Main model: {MODEL_MAIN}")
else:
    print("Check your OPENAI_API_KEY in .env")

## 5.2 Step 1 — LLM as Router (Link Selection)

The first agentic step: instead of writing brittle regex rules to decide which links are "about" or "careers" pages, we let an LLM make this nuanced judgment.

This is the **LLM-as-classifier** pattern &mdash; use language understanding for decisions that would require complex heuristics in traditional code.

In [ ]:
# ── Stage 5 ── Cell 2: LLM-Powered Link Intelligence ─────────────────────────

LINK_CLASSIFIER_SYSTEM = """You are a web content analyst. Given a list of URLs from a company website,
identify which pages would be most useful for creating a company intelligence report.

Select pages relevant to: About the company, Products/Services, Team/Leadership, 
Customers/Case studies, Careers/Culture, Investors/Press, Technology/Blog.

Exclude: Terms of Service, Privacy Policy, Login/Auth pages, social media links, 
CDN/static assets (images, fonts), email links.

CRITICAL: Return ONLY valid JSON — no text before or after.
Schema:
{
  "selected": [
    {"url": "full https url", "type": "about|product|team|customers|careers|press|blog", "priority": 1-5}
  ],
  "rationale": "one sentence explaining selection strategy"
}

Priority 1 = most important, 5 = nice to have. Select at most 8 URLs.
"""

def classify_links(base_url: str) -> dict:
    """
    Use an LLM to intelligently select which pages to read.
    This replaces fragile regex-based URL parsing with language understanding.
    """
    print(f"  Fetching links from {base_url}...")
    all_links = fetch_website_links(base_url)
    print(f"  Found {len(all_links)} raw links")
    
    user_prompt = f"""Base URL: {base_url}

All links found on this page:
{chr(10).join(all_links)}

Select the most relevant links for a company intelligence report."""
    
    response = openai.chat.completions.create(
        model           = MODEL_FAST,   # use cheapest model for this routing decision
        messages        = [
            {"role": "system", "content": LINK_CLASSIFIER_SYSTEM},
            {"role": "user",   "content": user_prompt},
        ],
        response_format = {"type": "json_object"},
        temperature     = 0.1,
    )
    
    result = json.loads(response.choices[0].message.content)
    selected = result.get("selected", [])
    print(f"  Selected {len(selected)} relevant pages ({response.usage.total_tokens} tokens)")
    if "rationale" in result:
        print(f"  Strategy: {result['rationale']}")
    return result

# Test it — use a real Indian startup or tech company
print("Testing link classifier on Zepto's website...")
link_data = classify_links("https://www.zeptonow.com")
print()
print("Selected pages:")
for page in link_data.get("selected", []):
    print(f"  [P{page['priority']}] {page['type']:<12} {page['url']}")

In [ ]:
# ── Stage 5 ── Cell 3: Smart Content Aggregator ──────────────────────────────

def aggregate_content(base_url: str, max_chars_per_page: int = 2000) -> dict:
    """
    Fetch and aggregate content from a company website intelligently.
    Returns structured data ready for report generation.
    """
    start_time = time.time()
    
    print(f"Aggregating content for: {base_url}")
    
    # Step 1: Get the homepage
    print("  [1/3] Fetching homepage...")
    homepage = fetch_website_contents(base_url)
    
    # Step 2: Classify links (LLM decision)
    print("  [2/3] Classifying links (LLM decision)...")
    link_data = classify_links(base_url)
    
    # Step 3: Fetch selected pages
    print("  [3/3] Fetching selected pages...")
    pages = {
        "homepage": {
            "url":     base_url,
            "type":    "homepage",
            "content": homepage[:max_chars_per_page],
            "chars":   min(len(homepage), max_chars_per_page),
        }
    }
    
    for page_info in sorted(link_data.get("selected", []), key=lambda x: x.get("priority", 5)):
        url      = page_info["url"]
        ptype    = page_info["type"]
        priority = page_info.get("priority", 5)
        
        try:
            content = fetch_website_contents(url)
            pages[ptype] = {
                "url":      url,
                "type":     ptype,
                "priority": priority,
                "content":  content[:max_chars_per_page],
                "chars":    min(len(content), max_chars_per_page),
            }
            print(f"    Fetched: {ptype} ({min(len(content), max_chars_per_page):,} chars)")
        except Exception as e:
            print(f"    Skip: {url} — {str(e)[:50]}")
    
    elapsed = time.time() - start_time
    total_chars = sum(p["chars"] for p in pages.values())
    total_tokens = len(enc.encode(" ".join(p["content"] for p in pages.values())))
    
    print(f"\nAggregation complete: {len(pages)} pages | {total_chars:,} chars | ~{total_tokens:,} tokens | {elapsed:.1f}s")
    
    return {
        "base_url":  base_url,
        "pages":     pages,
        "meta":      {"total_chars": total_chars, "total_tokens": total_tokens, "elapsed_s": elapsed},
    }

# Test
content_data = aggregate_content("https://www.zeptonow.com", max_chars_per_page=1500)

## 5.3 Step 2 — LLM as Generator (Report Creation with Streaming)

Now we combine all gathered content into a final generation call.

**Streaming** is critical for UX &mdash; users see output immediately instead of waiting for the full response. We'll stream the output and render it as live markdown.

In [ ]:
# ── Stage 5 ── Cell 4: Report Generation with Streaming ──────────────────────

REPORT_SYSTEM = """You are a senior venture capital analyst. 
Analyse startup website content and produce a structured diligence brief.

The brief must include these sections:
## Company Overview
One strong paragraph. What they do, who they serve, what problem they solve. Be direct.

## Product & Business Model  
Bullet points. Name specific products, revenue model, and pricing signals if visible.

## Market & Differentiation
What problem does this solve better than alternatives? What is their moat? 2-3 sharp bullets.

## Traction Signals
Growth indicators, customer logos, press mentions, hiring signals. Skip if absent.

## Team & Culture
Leadership signals from the website. Only include if evidence exists.

## VC Diligence Verdict
3-sentence synthesis written for a partner meeting: 
What stage is this company at? What's the opportunity? What's the key risk?

Rules:
- Write in professional, direct prose — like a memo to a GP
- No marketing fluff or vague adjectives
- If you don't have evidence for something, omit it rather than guess
- Markdown only, no code blocks"""

def build_report_prompt(company_name: str, content_data: dict) -> str:
    """Build the user prompt from aggregated content data."""
    parts = [f"Company: {company_name}", f"Website: {content_data['base_url']}", "\n---\n"]
    
    # Order: homepage first, then by priority
    pages = content_data["pages"]
    ordered = sorted(pages.values(), key=lambda p: (p.get("priority", 0) if p["type"] != "homepage" else -1))
    
    for page in ordered:
        parts.append(f"## {page['type'].title()} ({page['url']})")
        parts.append(page["content"])
        parts.append("\n")
    
    return "\n".join(parts)


def stream_report(company_name: str, base_url: str) -> str:
    """
    Full pipeline: aggregate content -> stream VC diligence brief.
    Returns the final report text.
    """
    print(f"Building intelligence report for: {company_name}\n")
    
    # Step 1: Aggregate (the agentic part)
    content_data = aggregate_content(base_url, max_chars_per_page=2000)
    
    # Step 2: Generate (the generative part, streamed)
    user_prompt  = build_report_prompt(company_name, content_data)
    token_count  = len(enc.encode(user_prompt))
    cost_estimate = token_count / 1_000_000 * 0.40
    
    print(f"\nGenerating report (~{token_count:,} input tokens, ~${cost_estimate:.4f})...")
    print("=" * 60)
    
    stream = openai.chat.completions.create(
        model    = MODEL_MAIN,
        messages = [
            {"role": "system", "content": REPORT_SYSTEM},
            {"role": "user",   "content": user_prompt},
        ],
        stream      = True,
        temperature = 0.3,
    )
    
    full_report    = ""
    display_handle = display(Markdown("*Generating report...*"), display_id=True)
    
    for chunk in stream:
        delta        = chunk.choices[0].delta.content or ""
        full_report += delta
        update_display(Markdown(full_report), display_id=display_handle.display_id)
    
    return full_report

# Generate a diligence brief for a real startup!
# Try any Indian or global startup — Razorpay, Zepto, Groww, Anthropic, etc.
report = stream_report("Razorpay", "https://razorpay.com")

In [ ]:
# ── Stage 5 ── Cell 5: Multi-Company Report Factory ─────────────────────────
# Real business use case: competitive intelligence briefing.

def compare_companies(companies: list, brief: bool = True) -> None:
    """
    Generate intelligence reports for multiple companies.
    
    Parameters:
    - companies: list of (name, url) tuples
    - brief: if True, generate shorter reports for speed
    """
    system = REPORT_SYSTEM if not brief else REPORT_SYSTEM + "\n\nBRIEF MODE: Keep each section to 1-2 bullet points. Total report under 300 words."
    
    reports = {}
    
    for name, url in companies:
        print(f"\n{'='*60}")
        print(f"  {name}")
        print(f"{'='*60}")
        
        try:
            content_data = aggregate_content(url, max_chars_per_page=1200)
            user_prompt  = build_report_prompt(name, content_data)
            
            response = openai.chat.completions.create(
                model    = MODEL_MAIN,
                messages = [
                    {"role": "system", "content": system},
                    {"role": "user",   "content": user_prompt},
                ],
                temperature = 0.2,
            )
            reports[name] = response.choices[0].message.content
            display(Markdown(f"### {name}\n" + reports[name]))
        
        except Exception as e:
            print(f"  Error processing {name}: {e}")
    
    return reports


# Try comparing two startups (uncomment)
# compare_companies([
#     ("Razorpay", "https://razorpay.com"),
#     ("Cashfree Payments", "https://cashfree.com"),
# ], brief=True)

## 5.4 The Full Architecture — Revisited

Let's now look at what we built and name the patterns explicitly:

```
classify_links()  ─→  LLM-as-Classifier (routing decision)
   ↓
aggregate_content()  ─→  Orchestrator (coordinates scraping + classification)
   ↓
build_report_prompt()  ─→  Prompt Builder (structures data for generation)
   ↓  
stream_report()  ─→  LLM-as-Generator (final output, streamed)
```

**What makes this "agentic":**
1. The pipeline makes autonomous decisions (which links to follow)
2. Multiple models are used for different subtasks (nano for routing, mini for generation)
3. External tools are called (web scraper)
4. Output is streamed progressively

**What would make this more agentic (Week 2+):**
- Add a verification step where the LLM checks if the report is complete
- Add web search to fill in gaps
- Add a feedback loop: if the report quality is low, re-fetch more pages

## Stage 5 Challenge

**Build a Startup Pitch Evaluator**

Combine what you've built into a tool that evaluates a startup's pitch from their website:

1. Takes: a startup URL + an investor thesis (text describing what you invest in)
2. Generates a VC diligence brief on the startup (Stage 5 pipeline)  
3. Uses the brief + investor thesis to evaluate fit
4. Generates a structured investment memo that:
   - Scores the startup on 5 dimensions (market, team, product, traction, fit)
   - Identifies the 3 key questions you'd ask in a partner meeting
   - Gives a final recommendation: Pass / Watch / Pursue
5. Streams the memo with live rendering

**Architecture hint:**
```
startup_url + investor_thesis  →  [aggregate_content]  →  startup_intelligence
startup_intelligence + thesis  →  [LLM: score_dimensions]  →  scores_json
scores_json + intelligence  →  [LLM: generate_memo, streamed]  →  investment_memo
```

This is a 3-call agentic pipeline — each LLM call has a distinct, focused purpose.

In [ ]:
# ── Stage 5 ── Challenge: Startup Pitch Evaluator ───────────────────────────

def evaluate_startup(
    startup_url:      str,
    investor_thesis:  str,
) -> str:
    """
    3-call agentic pipeline for VC-style startup evaluation.
    
    Call 1: aggregate_content(startup_url)  → startup intelligence
    Call 2: LLM scores startup on 5 dimensions → scores JSON
    Call 3: LLM generates investment memo, streamed
    """
    
    # ── CALL 1: Get startup intelligence ──────────────────────────────────────
    print("Step 1/3: Building startup intelligence...")
    content_data = aggregate_content(startup_url, max_chars_per_page=1500)
    startup_brief = build_report_prompt("Target Startup", content_data)
    
    # ── CALL 2: Score the startup ─────────────────────────────────────────────
    print("Step 2/3: Scoring startup dimensions...")
    
    # TODO: Design this prompt
    scoring_system = """TODO: You are a VC analyst scoring a startup.
Evaluate on 5 dimensions and return ONLY valid JSON — no preamble:
{
  "market_size": {"score": 1-10, "rationale": "one sentence"},
  "product_clarity": {"score": 1-10, "rationale": "one sentence"},
  "team_signals": {"score": 1-10, "rationale": "one sentence"},
  "traction_evidence": {"score": 1-10, "rationale": "one sentence"},
  "thesis_fit": {"score": 1-10, "rationale": "one sentence"},
  "partner_questions": ["question 1", "question 2", "question 3"],
  "overall": "Pass | Watch | Pursue"
}"""
    
    # TODO: Call the API here
    # score_response = openai.chat.completions.create(...)
    # scores = json.loads(score_response.choices[0].message.content)
    scores = {"overall": "TODO — implement Call 2"}
    
    # ── CALL 3: Generate investment memo (streamed) ────────────────────────────
    print("Step 3/3: Generating investment memo...\n")
    
    # TODO: Design this prompt and stream the output
    memo_system = """TODO: You are a VC writing a partner meeting memo.
Given the scores and startup intelligence, write a structured memo:
- 1-paragraph executive summary (what they do, stage, opportunity)
- Score card table (5 dimensions with scores and rationales)
- 3 key diligence questions for the partner meeting
- Final recommendation: Pass / Watch / Pursue with a 2-sentence justification
- Keep it under 400 words. No fluff."""
    
    # TODO: Build user prompt combining startup_brief + scores, then stream
    # user_prompt = f"Investor Thesis: {investor_thesis}\n\nStartup Brief:\n{startup_brief}\n\nScores: {json.dumps(scores, indent=2)}"
    # stream = openai.chat.completions.create(... stream=True ...)
    
    print("TODO: Implement the memo generation step with streaming")
    return "TODO"


# Test (uncomment when implemented)
# evaluate_startup(
#     startup_url     = "https://razorpay.com",
#     investor_thesis = """
#         We invest in B2B fintech infrastructure in India and Southeast Asia.
#         We look for: strong developer adoption, embedded finance angles,
#         clear path to profitability, and founding teams with deep domain expertise.
#     """,
# )

---
<div style="background: linear-gradient(135deg, #1a1a2e, #16213e); padding: 30px; border-radius: 12px; text-align: center;">
  <h1 style="color: #e94560; margin-bottom: 12px;">LLM Engineering  &mdash; 01 : Complete</h1>
  <p style="color: #ccc; font-size: 1.1em;">You have gone from zero to building a production-grade agentic AI pipeline.</p>
</div>

---

## What You Built

| Stage | You Built | Production Pattern |
|-------|-----------|-------------------|
| 1 | Cricket Match Analyst | Retrieval + Generation |
| 2 | Financial News Parser + Sentiment Engine | Structured Output + Format Control |
| 3 | Universal LLM Wrapper + Router | Provider Abstraction + Task Routing |
| 4 | Production Chatbot with Memory Management | Stateful Conversation + Context Compression |
| 5 | Startup Intelligence Radar | Multi-Step Agentic Pipeline + Streaming |

---

## The 7 Mental Models That Will Make You a Better AI Engineer

1. **LLMs are HTTP endpoints** — not magic boxes. Treat them like any other API.
2. **Prompts are programs** — version-control them, test them, review them like code.
3. **Statelessness is fundamental** — memory is always your job to implement.
4. **Tokens are the unit of currency** — always budget and estimate before running at scale.
5. **One interface, many providers** — never couple your code to a single model family.
6. **Use the right model for the job** — routing cheap tasks to cheap models cuts costs 10x.
7. **Agents = functions + decisions** — an agent is just code where an LLM makes some decisions.

---

## What's Coming in Week 2

| Topic | What You'll Build |
|-------|------------------|
| **Embeddings** | Semantic search over your own documents |
| **Vector Databases** | RAG (Retrieval-Augmented Generation) system |
| **Function Calling** | LLMs that can call your APIs |
| **Evaluation** | Automated testing framework for AI outputs |
| **Fine-tuning concepts** | When and why to customise a model |

---

## Quick Reference

```python
# The universal call pattern you should memorise:
from openai import OpenAI
client   = OpenAI()   # or OpenAI(base_url=..., api_key=...)  for other providers
response = client.chat.completions.create(
    model       = "gpt-4.1-mini",
    messages    = [
        {"role": "system",    "content": YOUR_SYSTEM_PROMPT},
        {"role": "user",      "content": USER_INPUT},
        # Add prior turns here for conversation memory
    ],
    temperature       = 0.7,              # 0=deterministic, 1+=creative
    max_tokens        = 500,             # cap output length
    response_format   = {"type": "json_object"},   # optional: forces JSON output
    stream            = True,            # optional: streaming response
)
# Single response:   response.choices[0].message.content
# Streaming:         for chunk in response: delta = chunk.choices[0].delta.content or ""
```

---

*Resources: OpenAI docs: [platform.openai.com/docs](https://platform.openai.com/docs) &mdash; Ollama: [ollama.com](https://ollama.com)*